#### **First Dataset**

In [22]:
import pandas as pd
xmas = pd.read_csv("https://www.dropbox.com/scl/fi/qxaslqqp5p08i1650rpc4/xmas.csv?rlkey=erdxi7jbh7pqf9fh4lv4cayp5&dl=1")

In [23]:
xmas.head()

,Day,Day.in.Words,Gift.Item,Verb,Adjective,Location
0,1,first,partridge,NaN,NaN,in a pear tree
1,2,second,dove,NaN,turtle,NaN
2,3,third,hen,NaN,french,NaN
3,4,fourth,bird,NaN,calling,NaN
4,5,fifth,ring,NaN,golden,NaN


#### **Practice making sing_day()**

In [24]:
#| code-fold: true
#| embed-resources: true

def sing_day(day, day_in_words, gifts):

    if day < 1 or day > 12:
        return "Try again"

    verse = f"On the {day_in_words} day of Christmas, my true love gave to me:\n{gifts}"
    return verse


In [25]:
print(sing_day(1, "first", "A partridge in a pear tree"))
print(sing_day(2, "second", "Two turtle doves, and a partridge in a pear tree"))

On the first day of Christmas, my true love gave to me:
A partridge in a pear tree
On the second day of Christmas, my true love gave to me:
Two turtle doves, and a partridge in a pear tree


#### **Pluralizing gift**

In [26]:
#| code-fold: true
#| embed-resources: true

def pluralize_gift(gift):
    if isinstance(gift, pd.Series):
        return gift.apply(pluralize_gift)

    if gift == "goose":
        gift = "geese"
    elif len(gift) >= 2 and gift.endswith("y") and gift[-2].lower() not in "aeiou":
        gift = gift[:-1] + "ies"
    else:
        gift = gift + "s"

    return gift

##### **TEST 1**

In [27]:
# Should work
pluralize_gift("goose")

# Will work if your function is vectorized!
pluralize_gift(xmas['Gift.Item'])

0     partridges
1          doves
2           hens
3          birds
4          rings
5          geese
6          swans
7          maids
8         ladies
9          lords
10        pipers
11      drummers
Name: Gift.Item, dtype: object

#### **Make_phrase function**

In [28]:
#| code-fold: true
#| embed-resources: true

def make_phrase(num, num_word, item, verb, adjective, location):
  """
  <documentation here>
  """

  verb = "" if pd.isna(verb) else verb
  adjective = "" if pd.isna(adjective) else adjective
  location = "" if pd.isna(location) else location

  if num > 1 and item.lower() not in ['lords', 'ladies', 'pipers', 'drummers']:
      item = pluralize_gift(item)

  starts_vowel = False
  if isinstance(item, str) and len(item) > 0:
      starts_vowel = item.lower()[0] in 'aeiou'

  day_prefix = ""
  if num == 1:
      if starts_vowel:
          day_prefix = "an"
      else:
          day_prefix = "a"
  else:
      day_prefix = num_word

  adjective_part = f"{adjective} " if adjective else ""
  verb_part = f" {verb}" if verb else ""
  location_part = f" {location}" if location else ""

  if num == 1:
      phrase = f"{day_prefix} {item}{location_part}"
  else:
       phrase = f"{day_prefix} {adjective_part}{item}{verb_part}{location_part}".strip()


  return phrase

In [29]:
make_phrase(num = 10,
            num_word = "ten",
            item = "lords",
            verb = "a-leaping",
            adjective = "",
            location = "")

'ten lords a-leaping'

In [30]:
pluralize_gift("cat")

make_phrase(num = 1,
            num_word= "first",
            item="cat",
            verb="",
            adjective="",
            location="in a pear tree")

'a cat in a pear tree'

In [31]:
pluralize_gift("drummer")

make_phrase(num = 12,
            num_word= "twelve",
            item="drummer",
            verb="drumming",
            adjective="",
            location="")

'twelve drummers drumming'

#### **Mapping Day.in.Word to the number spelt out**

In [32]:
#| code-fold: true
#| embed-resources: true

xmas["write_day"] = xmas["Day.in.Words"].map({
    "first": "One",
    "second": "Two",
    "third": "Three",
    "fourth": "Four",
    "fifth": "Five",
    "sixth": "Six",
    "seventh": "Seven",
    "eighth": "Eight",
    "ninth": "Nine",
    "tenth": "Ten",
    "eleventh": "Eleven",
    "twelfth": "Twelve",
})
xmas.head()

,Day,Day.in.Words,Gift.Item,Verb,Adjective,Location,write_day
0,1,first,partridge,NaN,NaN,in a pear tree,One
1,2,second,dove,NaN,turtle,NaN,Two
2,3,third,hen,NaN,french,NaN,Three
3,4,fourth,bird,NaN,calling,NaN,Four
4,5,fifth,ring,NaN,golden,NaN,Five


In [33]:
#| code-fold: true
#| embed-resources: true

xmas['Full.Phrase'] = xmas.apply(
    lambda row: make_phrase(
        num=row['Day'],
        num_word=row['write_day'],
        item=row['Gift.Item'],
        verb=row['Verb'],
        adjective=row['Adjective'],
        location=row['Location']
    ),
    axis=1
)

display(xmas)

,Day,Day.in.Words,Gift.Item,Verb,Adjective,Location,write_day,Full.Phrase
0,1,first,partridge,NaN,NaN,in a pear tree,One,a partridge in a pear tree
1,2,second,dove,NaN,turtle,NaN,Two,Two turtle doves
2,3,third,hen,NaN,french,NaN,Three,Three french hens
3,4,fourth,bird,NaN,calling,NaN,Four,Four calling birds
4,5,fifth,ring,NaN,golden,NaN,Five,Five golden rings
5,6,sixth,goose,a-laying,NaN,NaN,Six,Six geese a-laying
6,7,seventh,swan,a-swimming,NaN,NaN,Seven,Seven swans a-swimming
7,8,eighth,maid,a-milking,NaN,NaN,Eight,Eight maids a-milking
8,9,ninth,lady,dancing,NaN,NaN,Nine,Nine ladies dancing
9,10,tenth,lord,a-leaping,NaN,NaN,Ten,Ten lords a-leaping


#### **Creating the sing_day() bringing it all together**

In [34]:
#| code-fold: true
#| embed-resources: true

def sing_day(dataset, num, phrase_col):
  """
  <documentation>
  """

  # Step 1: Setup the intro line
  # convert "1" to "first" etc.
  num_word = dataset.loc[num - 1, 'Day.in.Words']
  intro = "On the " + num_word + " day of Christmas, my true love sent to me: \n"

  # Step 2: Sing the gift phrases
  # Hint: What order are they gifts sung in each day?
  gifts = ""
  for i in range(num-1, -1, -1):
    gifts = gifts + dataset.loc[i, phrase_col]

    if i == 0:
            gifts = gifts + ".\n"
    else:
            gifts = gifts + ",\n"

    if num > 1:
        gifts = gifts.replace(dataset.loc[0, phrase_col],
                              "and " + dataset.loc[0, phrase_col], 1)


  # Step 3: Put it all together and return
  return intro + gifts

In [35]:
sing_day(xmas, 3, "Full.Phrase")

'On the third day of Christmas, my true love sent to me: \nThree french hens,\nTwo turtle doves,\nand a partridge in a pear tree.\n'

In [36]:
print(sing_day(xmas, 3, "Full.Phrase"))

On the third day of Christmas, my true love sent to me: 
Three french hens,
Two turtle doves,
and a partridge in a pear tree.



In [ ]:
complete_song_lyrics = print(sing_day(xmas, 12, "Full.Phrase"))
#was missing an iteration that would repeat the song stanza by stanza such that it does it from the top for each day

On the twelfth day of Christmas, my true love sent to me: 
Twelve drummers drumming,
Eleven pipers piping,
Ten lords a-leaping,
Nine ladies dancing,
Eight maids a-milking,
Seven swans a-swimming,
Six geese a-laying,
Five golden rings,
Four calling birds,
Three french hens,
Two turtle doves,
and a partridge in a pear tree.



#### **Second dataset**

In [20]:
xmas2 = pd.read_csv("https://www.dropbox.com/scl/fi/p9x9k8xwuzs9rhp582vfy/xmas_2.csv?rlkey=kvc3j3lmyn4opcidsrhcmrof1&dl=1")

In [ ]:
##complete_song_lyrics2 = print(sing_day(xmas2, 12, "Full.Phrase"))
#Should have each code block for xmas 2 as xmas one to have similar dataset

KeyError: 'Full.Phrase'

#### **First, we need to map the day in words to a number to be used in our function and then we need to create a column for the full phrase. Pluralization was applied as well.**

In [19]:
#| code-fold: true
#| embed-resources: true

xmas2["write_day"] = xmas2["Day.in.Words"].map({
    "first": "One",
    "second": "Two",
    "third": "Three",
    "fourth": "Four",
    "fifth": "Five",
    "sixth": "Six",
    "seventh": "Seven",
    "eighth": "Eight",
    "ninth": "Nine",
    "tenth": "Ten",
    "eleventh": "Eleven",
    "twelfth": "Twelve",
})
display(xmas2)

NameError: name 'xmas2' is not defined

In [ ]:
#| code-fold: true
#| embed-resources: true

xmas2['Full.Phrase'] = xmas2.apply(
    lambda row: make_phrase(
        num=row['Day'],
        num_word=row['write_day'],
        item=row['Gift.Item'],
        verb=row['Verb'],
        adjective=row['Adjective'],
        location=row['Location']
    ),
    axis=1
)

display(xmas2)

,Day,Day.in.Words,Gift.Item,Verb,Adjective,Location,write_day,Full.Phrase
0,1,first,email,NaN,NaN,from Cal Poly,One,an email from Cal Poly
1,2,second,point,NaN,meal,NaN,Two,Two meal points
2,3,third,pen,NaN,lost,NaN,Three,Three lost pens
3,4,fourth,review,NaN,course,NaN,Four,Four course reviews
4,5,fifth,exam,NaN,practice,NaN,Five,Five practice exams
5,6,sixth,grader,grading,NaN,NaN,Six,Six graders grading
6,7,seventh,senior,stressing,NaN,NaN,Seven,Seven seniors stressing
7,8,eighth,mom,a-calling,NaN,NaN,Eight,Eight moms a-calling
8,9,ninth,party,bumping,NaN,NaN,Nine,Nine parties bumping
9,10,tenth,load,of laundry,NaN,NaN,Ten,Ten loads of laundry


In [212]:
complete_song_lyrics2 = print(sing_day(xmas2, 12, "Full.Phrase"))

On the twelfth day of Christmas, my true love sent to me: 
Twelve hours sleeping,
Eleven friends goodbye-ing,
Ten loads of laundry,
Nine parties bumping,
Eight moms a-calling,
Seven seniors stressing,
Six graders grading,
Five practice exams,
Four course reviews,
Three lost pens,
Two meal points,
and an email from Cal Poly.

